In [82]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

In [83]:
#%cd ../..
#!ls

In [84]:
tmp_data = pd.read_csv('EDA/data/findata.csv', index_col=0)

In [85]:
tmp_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   float64
 13  MaxPartialCharge     1001 non-null   float64
 14  MinPartialCharge     1001 non-null   float64
 15  MaxAbsPartialCharge  1001 non-null   float6

In [86]:
go_data = tmp_data.copy()

In [87]:
# Удалим не нужные тут таргеты 
go_data['bin_target'] = (go_data['CC50, mM'] > go_data['CC50, mM'].median()).astype(int)

go_data = go_data.drop(columns=['IC50, mM','CC50, mM', 'SI'])


In [88]:
# Выбираем самые полезные параметры 

# Рассчитываем корреляцию всех признаков 
go_correlations = go_data.corr()['bin_target'].abs().sort_values()

# Отбираем признаки с корреляцией больше 0.1 
gl_high_info_features = go_correlations[go_correlations > 0.1]

print("Информативные признаки (есть связь):")
gl_high_info_features = gl_high_info_features.drop(['bin_target'], errors='ignore')


#Для финальной модели оставляем только информативные параметры  
gl_final_param = gl_high_info_features.index.unique().tolist() 

print(len(gl_final_param))
display(go_correlations.sort_values(ascending=False).head(50))

display(gl_final_param)

Информативные признаки (есть связь):
19


bin_target             1.000000
PEOE_VSA7              0.213665
VSA_EState4            0.188786
BCUT2D_CHGLO           0.166647
SMR_VSA5               0.159891
FpDensityMorgan2       0.155461
AvgIpc                 0.148534
SlogP_VSA5             0.145791
FpDensityMorgan3       0.136385
FpDensityMorgan1       0.135166
PEOE_VSA6              0.134295
EState_VSA3            0.133097
PEOE_VSA8              0.129057
BCUT2D_MWLOW           0.127481
PEOE_VSA9              0.113484
Chi4n                  0.110164
Chi4v                  0.107445
MaxPartialCharge       0.107016
BCUT2D_LOGPLOW         0.101100
SlogP_VSA2             0.100547
MinAbsPartialCharge    0.098306
Kappa3                 0.096522
Chi2n                  0.096301
VSA_EState7            0.095162
MinEStateIndex         0.091382
Chi2v                  0.089121
NumHeteroatoms         0.079343
MolLogP                0.077454
SMR_VSA1               0.073515
EState_VSA4            0.070585
MaxAbsEStateIndex      0.068605
MaxEStat

['SlogP_VSA2',
 'BCUT2D_LOGPLOW',
 'MaxPartialCharge',
 'Chi4v',
 'Chi4n',
 'PEOE_VSA9',
 'BCUT2D_MWLOW',
 'PEOE_VSA8',
 'EState_VSA3',
 'PEOE_VSA6',
 'FpDensityMorgan1',
 'FpDensityMorgan3',
 'SlogP_VSA5',
 'AvgIpc',
 'FpDensityMorgan2',
 'SMR_VSA5',
 'BCUT2D_CHGLO',
 'VSA_EState4',
 'PEOE_VSA7']

In [89]:
#перебором выявим параметры котрые коррелируют между собой > 60% и оставим только второй 

for col_1 in gl_final_param:
    for col_2 in gl_final_param:
        if col_1 != col_2:
            lv_correlation = go_data[col_1].corr(go_data[col_2])
            if lv_correlation >= 0.70:
                print(f' Параметр {col_1} коррелирует с парамтером {col_2} : {lv_correlation}')
                gl_final_param.remove(col_2)
display(gl_final_param)

 Параметр BCUT2D_LOGPLOW коррелирует с парамтером BCUT2D_CHGLO : 0.8673416403304163
 Параметр Chi4v коррелирует с парамтером Chi4n : 0.9663789355818357
 Параметр FpDensityMorgan1 коррелирует с парамтером FpDensityMorgan3 : 0.7973776069503877
 Параметр FpDensityMorgan1 коррелирует с парамтером FpDensityMorgan2 : 0.9477566410321775
 Параметр SlogP_VSA5 коррелирует с парамтером SMR_VSA5 : 0.8633308881356347


['SlogP_VSA2',
 'BCUT2D_LOGPLOW',
 'MaxPartialCharge',
 'Chi4v',
 'PEOE_VSA9',
 'BCUT2D_MWLOW',
 'PEOE_VSA8',
 'EState_VSA3',
 'PEOE_VSA6',
 'FpDensityMorgan1',
 'SlogP_VSA5',
 'AvgIpc',
 'VSA_EState4',
 'PEOE_VSA7']

In [90]:
go_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 77 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   MaxAbsEStateIndex    1001 non-null   float64
 1   MaxEStateIndex       1001 non-null   float64
 2   MinAbsEStateIndex    1001 non-null   float64
 3   MinEStateIndex       1001 non-null   float64
 4   qed                  1001 non-null   float64
 5   SPS                  1001 non-null   float64
 6   MolWt                1001 non-null   float64
 7   HeavyAtomMolWt       1001 non-null   float64
 8   ExactMolWt           1001 non-null   float64
 9   NumValenceElectrons  1001 non-null   float64
 10  MaxPartialCharge     1001 non-null   float64
 11  MinPartialCharge     1001 non-null   float64
 12  MaxAbsPartialCharge  1001 non-null   float64
 13  MinAbsPartialCharge  1001 non-null   float64
 14  FpDensityMorgan1     1001 non-null   float64
 15  FpDensityMorgan2     1001 non-null   float6

In [91]:
X = go_data[gl_final_param]
#X = go_data.drop(columns='IC50, mM')
y = go_data['bin_target']

In [92]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (700, 14), (700,)
Test dataset size: (301, 14), (301,)


In [93]:
print(go_data['bin_target'].value_counts())

bin_target
0    502
1    499
Name: count, dtype: int64


In [94]:
import warnings
warnings.filterwarnings('ignore')

In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

models = {
    "LogisticRegression": (LogisticRegression(max_iter=1000), 
        {
            'C': [0.1, 1.0, 10.0], 
            'penalty': ['l2']
        }),

    "RandomForestClassifier": (RandomForestClassifier(), 
        {
            'n_estimators': [50, 150, 350],
            'max_depth': [None, 5, 10]
        }),

    "CatBoostClassifier": (CatBoostClassifier(), 
        {
            'iterations': [50, 150, 350],
            'learning_rate': [0.1, 0.5, 0.7],
            'verbose': [0]

        })
    
}



In [96]:
from skopt import BayesSearchCV
from sklearn.metrics import silhouette_score
from sklearn.model_selection import PredefinedSplit

# Перебор моделей
best_global_score = -10
best_model = None
results_report = []


for name, (model, params) in models.items():
    print(f"Обучаем {name}...")

    # Байесовская оптимизация гиперпараметров
    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=params,
        n_iter=35,
        cv=15,
        scoring='roc_auc',  
        n_jobs=-1,
        random_state=42
    )

    # Обучение модели
    bayes_search.fit(X_train, y_train)

    # 
    score = bayes_search.best_score_  # type: ignore
    results_report.append({"Model": name, "Score": score, "Params": bayes_search.best_params_}) # type: ignore
    
    # Сохраняем абсолютного победителя
    if score > best_global_score:
        best_global_score = score
        best_model = bayes_search.best_estimator_ # type: ignore

# --- АНАЛИЗ ---
print("\n--- Report  ---")
print(pd.DataFrame(results_report))
print(f"\n Лучшая модель: {best_model}")



Обучаем LogisticRegression...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

Обучаем RandomForestClassifier...
Обучаем CatBoostClassifier...

--- Report  ---
                    Model     Score  \
0      LogisticRegression  0.716958   
1  RandomForestClassifier  0.828794   
2      CatBoostClassifier  0.828768   

                                              Params  
0                        {'C': 0.1, 'penalty': 'l2'}  
1             {'max_depth': 10, 'n_estimators': 150}  
2  {'iterations': 150, 'learning_rate': 0.5, 'ver...  

 Лучшая модель: RandomForestClassifier(max_depth=10, n_estimators=150)


In [97]:
# на тестовой выборке 
from sklearn import metrics

y_pred = best_model.predict(X_test)  # type: ignore

print("MAE", metrics.mean_absolute_error(y_test, y_pred))
print("MSE", metrics.mean_squared_error(y_test, y_pred))
print("R2 Score:", best_model.score(X_test, y_test)) # type: ignore

MAE 0.27906976744186046
MSE 0.27906976744186046
R2 Score: 0.7209302325581395
